# Temporal Deforestation Detection — Sklearn MLP Pipeline
Loads 6 years of AEF embeddings per tile, decodes RADD + GLAD-S2 labels vectorially,
builds a dataset, trains an MLP, and outputs polygon predictions with month estimates.

In [46]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install rasterio geopandas scikit-learn tqdm numpy matplotlib pandas scipy shapely --quiet


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from pathlib import Path
from tqdm.auto import tqdm
import geopandas as gpd

In [48]:
# ── Cell 3: Constants & dequantization (straight from workshop) ────────────────
POWER  = 2.0
SCALE  = 127.5
OFFSET = 127

def set_nan_on_zero_groups(emb, group_size=64, inplace=True):
    out = emb if inplace else emb.copy()
    grouped = out.reshape(out.shape[0] // group_size, group_size, -1)
    zero_groups = np.all(grouped == 0, axis=1)
    grouped[:] = np.where(zero_groups[:, None, :], np.nan, grouped)
    return out

def dequantize_embeddings(emb):
    out = emb.astype(np.float32)
    set_nan_on_zero_groups(out, group_size=64, inplace=True)
    out = (out - OFFSET) / SCALE
    neg = out < 0
    out = np.abs(out) ** POWER
    out[neg] *= -1
    return out

In [ ]:
# ── Cell 4: Data paths & discover all tiles ────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from config import AEF_TRAIN, AEF_TEST, RADD_DIR, GLADS2_DIR

YEARS = ['2020', '2021', '2022', '2023', '2024', '2025']

_radd_dir   = Path(RADD_DIR)
_glads2_dir = Path(GLADS2_DIR)

def get_tile_paths(tile_id, split='train'):
    paths = {}
    if split == 'train':
        paths['radd']        = _radd_dir   / f'radd_{tile_id}_labels.tif'
        paths['glads2']      = _glads2_dir / f'glads2_{tile_id}_alert.tif'
        paths['glads2_date'] = _glads2_dir / f'glads2_{tile_id}_alertDate.tif'
    aef_dir = AEF_TRAIN if split == 'train' else AEF_TEST
    for year in YEARS:
        paths[f'aef_{year}'] = Path(aef_dir) / f'{tile_id}_{year}.tiff'
    return paths

# Discover tiles from AEF train directory
aef_train_dir = Path(AEF_TRAIN)
ALL_TILE_IDS  = sorted(set(
    p.stem.rsplit('_', 1)[0] for p in aef_train_dir.glob('*.tiff')
))

# Split into full-consensus vs RADD-only (GLAD-S2 absent for some tiles)
valid_tiles     = []  # RADD + GLAD-S2 + 6 AEF years
radd_only_tiles = []  # RADD + 6 AEF years (no GLAD-S2)

for tid in ALL_TILE_IDS:
    paths   = get_tile_paths(tid)
    aef_ok  = all(paths[f'aef_{y}'].exists() for y in YEARS)
    radd_ok = paths['radd'].exists()
    g_ok    = paths['glads2'].exists() and paths['glads2_date'].exists()
    if aef_ok and radd_ok and g_ok:
        valid_tiles.append(tid)
    elif aef_ok and radd_ok:
        radd_only_tiles.append(tid)

all_train_tiles = valid_tiles + radd_only_tiles

print(f'Full consensus tiles (RADD + GLAD-S2): {len(valid_tiles)}')
print(f'RADD-only tiles                       : {len(radd_only_tiles)}')
print(f'Total training tiles                  : {len(all_train_tiles)}')
print(f'Feature dims per pixel: {len(YEARS)} years × 64 = {len(YEARS) * 64}')

TILE_ID = all_train_tiles[0]


In [50]:
# ── Cell 5: Load full temporal AEF stack for one tile ─────────────────────────
def load_aef_stack(tile_id, split='train'):
    """Returns (6, 64, H, W) float32 — one 64-dim embedding per year per pixel."""
    paths = get_tile_paths(tile_id, split)
    stack = []
    transform, crs, shape = None, None, None
    for year in YEARS:
        with rasterio.open(paths[f'aef_{year}']) as src:
            raw = src.read()                        # (64, H, W) uint8
            if transform is None:
                transform = src.transform
                crs       = src.crs
                shape     = src.shape
        stack.append(dequantize_embeddings(raw))    # (64, H, W) float32
    return np.stack(stack, axis=0), transform, crs, shape  # (6, 64, H, W)

In [ ]:
# ── Cell 6: Build binary label + month label for one tile (vectorised) ─────────
EPOCH_RADD   = datetime.date(2014, 12, 31)
EPOCH_GLADS2 = datetime.date(2019,  1,  1)

def days_to_month_index(days_arr, epoch):
    """Vectorised: days-since-epoch -> month index 0-71 (2020-01 to 2025-12), -1 outside."""
    dates    = pd.to_datetime(epoch.isoformat()) + pd.to_timedelta(days_arr.ravel().astype(int), unit='D')
    years    = dates.year.values.reshape(days_arr.shape)
    months   = dates.month.values.reshape(days_arr.shape)
    in_range = (years >= 2020) & (years <= 2025)
    return np.where(in_range, (years - 2020) * 12 + (months - 1), -1).astype(np.int16)

def load_labels(tile_id, aef_transform, aef_crs, aef_shape, split='train'):
    """
    Returns:
        binary_mask  (H, W) uint8  -- 1 where consensus deforestation
        month_map    (H, W) int16  -- 0-71 = which month, -1 = no deforestation
        no_alert     (H, W) uint8  -- 1 where all sources agree: no deforestation

    Uses RADD + GLAD-S2 consensus when GLAD-S2 is available,
    falls back to RADD-only otherwise.
    """
    paths = get_tile_paths(tile_id, split)

    def reproj(arr, src_crs, src_transform, fill=-1):
        out = np.full(aef_shape, fill, dtype=arr.dtype)
        reproject(source=arr, destination=out,
                  src_transform=src_transform, src_crs=src_crs,
                  dst_transform=aef_transform, dst_crs=aef_crs,
                  resampling=Resampling.nearest)
        return out

    # ── RADD ──────────────────────────────────────────────────────────────────
    with rasterio.open(paths['radd']) as src:
        radd_raw       = src.read(1).astype(np.int32)
        radd_crs       = src.crs
        radd_transform = src.transform

    radd_conf  = radd_raw // 10000
    radd_days  = radd_raw % 10000
    radd_bin   = ((radd_raw > 0) & (radd_conf >= 3)).astype(np.uint8)
    radd_month = days_to_month_index(np.where(radd_bin, radd_days, 0), EPOCH_RADD)
    radd_month[~radd_bin.astype(bool)] = -1
    radd_bin   = reproj(radd_bin,   radd_crs, radd_transform, fill=0)
    radd_month = reproj(radd_month, radd_crs, radd_transform, fill=-1)

    # ── GLAD-S2 (optional) ────────────────────────────────────────────────────
    has_glads2 = (paths.get('glads2') and paths['glads2'].exists()
                  and paths.get('glads2_date') and paths['glads2_date'].exists())

    if has_glads2:
        with rasterio.open(paths['glads2']) as src:
            glads2_conf      = src.read(1).astype(np.uint8)
            glads2_crs       = src.crs
            glads2_transform = src.transform
        with rasterio.open(paths['glads2_date']) as src:
            glads2_days = src.read(1).astype(np.int32)

        glads2_bin   = ((glads2_conf >= 3) & (glads2_days > 365)).astype(np.uint8)
        glads2_month = days_to_month_index(np.where(glads2_bin, glads2_days, 0), EPOCH_GLADS2)
        glads2_month[~glads2_bin.astype(bool)] = -1
        glads2_bin   = reproj(glads2_bin,   glads2_crs, glads2_transform, fill=0)
        glads2_month = reproj(glads2_month, glads2_crs, glads2_transform, fill=-1)

        # Consensus: both sources must agree
        binary_mask = ((radd_bin + glads2_bin) >= 2).astype(np.uint8)
        no_alert    = ((radd_bin + glads2_bin) == 0).astype(np.uint8)

        month_map = np.full(aef_shape, -1, dtype=np.int16)
        both = (radd_bin == 1) & (glads2_bin == 1)
        one  = (radd_bin != glads2_bin)
        month_map[both] = ((radd_month[both].astype(np.int32) +
                            glads2_month[both].astype(np.int32)) // 2).astype(np.int16)
        month_map[one & (radd_bin == 1)]   = radd_month  [one & (radd_bin == 1)]
        month_map[one & (glads2_bin == 1)] = glads2_month[one & (glads2_bin == 1)]
    else:
        # RADD-only fallback for tiles without GLAD-S2
        binary_mask = radd_bin
        no_alert    = (radd_bin == 0).astype(np.uint8)
        month_map   = radd_month

    return binary_mask, month_map, no_alert


In [52]:
# ── Cell 7: Verify on single tile ─────────────────────────────────────────────
aef_stack, aef_transform, aef_crs, aef_shape = load_aef_stack(TILE_ID)
binary_mask, month_map, no_alert = load_labels(
    TILE_ID, aef_transform, aef_crs, aef_shape)

print(f'AEF stack shape  : {aef_stack.shape}')   # (6, 64, H, W)
print(f'Deforested px    : {binary_mask.sum():,}  ({100*binary_mask.mean():.1f}%)')
print(f'No-alert px      : {no_alert.sum():,}')
if (month_map >= 0).any():
    print(f'Month range      : {month_map[month_map>=0].min()} to {month_map[month_map>=0].max()}')

AEF stack shape  : (6, 64, 1004, 998)
Deforested px    : 290,234  (29.0%)
No-alert px      : 566,789
Month range      : 0 to 71


In [ ]:
# ── Cell 8: Build training dataset — capped per tile for speed ─────────────────
MAX_POS_PER_TILE = 5000   # cap so each tile contributes at most 5k positives
N_NEG_MULT       = 2      # 2 negatives per positive

all_X, all_y_bin, all_y_month = [], [], []

for tile_id in tqdm(all_train_tiles, desc='Building dataset'):
    stack, t, crs, shape = load_aef_stack(tile_id)
    binary, months, no_alert = load_labels(tile_id, t, crs, shape)

    # Flatten: (6, 64, H, W) -> (H*W, 384)
    flat = stack.reshape(len(YEARS) * 64, -1).T

    pos_idx = np.where(binary.ravel() == 1)[0]
    neg_idx = np.where(no_alert.ravel() == 1)[0]

    # Drop NaN pixels
    pos_idx = pos_idx[np.isfinite(flat[pos_idx]).all(axis=1)]
    neg_idx = neg_idx[np.isfinite(flat[neg_idx]).all(axis=1)]

    if len(pos_idx) == 0:
        continue

    rng = np.random.default_rng(42)

    if len(pos_idx) > MAX_POS_PER_TILE:
        pos_idx = rng.choice(pos_idx, MAX_POS_PER_TILE, replace=False)

    n_neg   = min(len(neg_idx), len(pos_idx) * N_NEG_MULT)
    neg_idx = rng.choice(neg_idx, n_neg, replace=False)

    idx = np.concatenate([pos_idx, neg_idx])
    all_X.append(flat[idx])
    all_y_bin.append(binary.ravel()[idx])
    all_y_month.append(months.ravel()[idx])

X       = np.vstack(all_X)
y_bin   = np.concatenate(all_y_bin).astype(np.int32)
y_month = np.concatenate(all_y_month).astype(np.int16)

print(f'Dataset shape    : {X.shape}')
print(f'Positives        : {y_bin.sum():,}  ({100*y_bin.mean():.1f}%)')
print(f'Negatives        : {(y_bin==0).sum():,}')
if (y_month >= 0).any():
    print(f'Month label range: {y_month[y_month>=0].min()} to {y_month[y_month>=0].max()}')


In [54]:
# ── Cell 9: Train temporal MLP ─────────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_val, y_train, y_val = train_test_split(
    X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp',    MLPClassifier(
                   hidden_layer_sizes=(256, 128, 64),
                   activation='relu',
                   max_iter=50,
                   early_stopping=True,
                   validation_fraction=0.1,
                   random_state=42,
                   verbose=True))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred,
      target_names=['no deforestation', 'deforestation']))

Iteration 1, loss = 0.09420627
Validation score: 0.976042
Iteration 2, loss = 0.07079757
Validation score: 0.977917
Iteration 3, loss = 0.06257490
Validation score: 0.977708
Iteration 4, loss = 0.05735222
Validation score: 0.977812
Iteration 5, loss = 0.05242753
Validation score: 0.976979
Iteration 6, loss = 0.04782188
Validation score: 0.978750
Iteration 7, loss = 0.04446305
Validation score: 0.980208
Iteration 8, loss = 0.04021093
Validation score: 0.979688
Iteration 9, loss = 0.03755334
Validation score: 0.980104
Iteration 10, loss = 0.03398443
Validation score: 0.980104
Iteration 11, loss = 0.03235402
Validation score: 0.979896
Iteration 12, loss = 0.02901965
Validation score: 0.979792
Iteration 13, loss = 0.02606392
Validation score: 0.980729
Iteration 14, loss = 0.02445242
Validation score: 0.979792
Iteration 15, loss = 0.02318727
Validation score: 0.978021
Iteration 16, loss = 0.02026326
Validation score: 0.981563
Iteration 17, loss = 0.01992279
Validation score: 0.980729
Iterat

In [ ]:
# ── Cell 10: Inference -> binary raster -> submission GeoJSON ────────────────────
import json, tempfile
from scipy import ndimage

# submission_utils lives at the project root
if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))
from submission_utils import raster_to_geojson

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

def predict_tile(tile_id, split='train'):
    """Returns binary pred_map, prob_map, and geospatial metadata."""
    stack, t, crs, shape = load_aef_stack(tile_id, split)
    flat = stack.reshape(len(YEARS) * 64, -1).T
    valid = np.isfinite(flat).all(axis=1)
    proba = np.zeros(flat.shape[0])
    proba[valid] = model.predict_proba(flat[valid])[:, 1]
    prob_map = proba.reshape(shape)
    pred_map = (prob_map > 0.5).astype(np.uint8)
    return pred_map, prob_map, t, crs, shape

def save_binary_raster(pred_map, transform, crs, out_path):
    with rasterio.open(
        out_path, 'w', driver='GTiff',
        height=pred_map.shape[0], width=pred_map.shape[1],
        count=1, dtype='uint8', crs=crs, transform=transform
    ) as dst:
        dst.write(pred_map.astype('uint8'), 1)

def add_time_steps(geojson_dict, pred_map, month_map, transform, native_crs):
    """Enrich each polygon with a time_step (YYYY-MM) from the month_map."""
    import geopandas as gpd
    from shapely.geometry import shape as shp
    from rasterio.features import geometry_mask

    H, W = pred_map.shape
    geoms_4326 = [shp(f['geometry']) for f in geojson_dict.get('features', [])]
    if not geoms_4326:
        return geojson_dict

    gdf = gpd.GeoDataFrame(geometry=geoms_4326, crs='EPSG:4326')
    gdf_native = gdf.to_crs(native_crs)

    for i, feat in enumerate(geojson_dict['features']):
        geom_native = gdf_native.geometry.iloc[i]
        try:
            mask = geometry_mask(
                [geom_native.__geo_interface__],
                transform=transform, invert=True, out_shape=(H, W)
            )
            valid_m = month_map[mask]
            valid_m = valid_m[valid_m >= 0]
            if len(valid_m):
                pm = int(np.median(valid_m))
                yr, mo = 2020 + pm // 12, 1 + pm % 12
                feat['properties']['time_step'] = f'{yr}-{mo:02d}'
            else:
                feat['properties']['time_step'] = None
        except Exception:
            feat['properties']['time_step'] = None

    return geojson_dict

# ── Run inference on all training tiles ───────────────────────────────────────
for tile_id in tqdm(all_train_tiles, desc='Inference'):
    pred_map, prob_map, t, crs, shape = predict_tile(tile_id)
    _, month_map, _ = load_labels(tile_id, t, crs, shape)

    bin_path = OUTPUT_DIR / f'{tile_id}_pred_binary.tif'
    save_binary_raster(pred_map, t, crs, bin_path)

    geojson_path = OUTPUT_DIR / f'{tile_id}_submission.geojson'
    try:
        geojson = raster_to_geojson(bin_path, output_path=None, min_area_ha=0.5)
        geojson = add_time_steps(geojson, pred_map, month_map, t, crs)
        with open(geojson_path, 'w') as fh:
            json.dump(geojson, fh)
        print(f'{tile_id}: {len(geojson["features"]):,} polygons saved')
    except ValueError as e:
        print(f'{tile_id}: skipped — {e}')

print(f'\nAll outputs in {OUTPUT_DIR}/')


In [ ]:
# ── Cell 12: Generate FINAL submission GeoJSON for test tiles ──────────────────
import json, tempfile
from pathlib import Path
from tqdm.auto import tqdm

if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))
from submission_utils import raster_to_geojson
from config import AEF_TEST

SUBMISSION_DIR = Path('submission')
SUBMISSION_DIR.mkdir(exist_ok=True)

def estimate_month_map(aef_stack):
    """Estimate deforestation month from the year of largest AEF change."""
    n_years, n_dims, H, W = aef_stack.shape
    diffs = np.stack([
        np.sqrt(np.sum((aef_stack[y + 1] - aef_stack[y]) ** 2, axis=0))
        for y in range(n_years - 1)
    ], axis=0)
    change_year_idx = np.argmax(diffs, axis=0)
    year_deforested = 2021 + change_year_idx
    return ((year_deforested - 2020) * 12 + 5).astype(np.int16)  # mid-year estimate

def enrich_and_validate(geojson_dict, month_map_est, transform, native_crs):
    import geopandas as gpd
    from shapely.geometry import shape as shp
    from rasterio.features import geometry_mask

    features = geojson_dict.get('features', [])
    if not features:
        return []

    geoms_4326 = [shp(f['geometry']) for f in features]
    gdf = gpd.GeoDataFrame(geometry=geoms_4326, crs='EPSG:4326')
    gdf_native = gdf.to_crs(native_crs)

    valid_features = []
    for i, feat in enumerate(features):
        if feat['geometry']['type'] not in ['Polygon', 'MultiPolygon']:
            continue
        geom_native = gdf_native.geometry.iloc[i]
        try:
            mask = geometry_mask(
                [geom_native.__geo_interface__],
                transform=transform, invert=True, out_shape=month_map_est.shape,
            )
            valid_m = month_map_est[mask]
            valid_m = valid_m[valid_m >= 0]
            if len(valid_m):
                pm = int(np.median(valid_m))
                yr, mo = 2020 + pm // 12, 1 + pm % 12
                feat['properties'] = {'time_step': f'{yr}-{mo:02d}'}
            else:
                feat['properties'] = {'time_step': None}
        except Exception:
            feat['properties'] = {'time_step': None}
        valid_features.append(feat)
    return valid_features

# Discover test tiles from AEF test directory
test_aef_dir = Path(AEF_TEST)
TEST_TILES   = sorted(set(p.stem.rsplit('_', 1)[0] for p in test_aef_dir.glob('*.tif*')))
print(f'Test tiles found: {len(TEST_TILES)}')

all_submission_features = []

for tile_id in tqdm(TEST_TILES, desc='Submission Inference'):
    stack, t, crs, shape = load_aef_stack(tile_id, split='test')
    flat = stack.reshape(len(YEARS) * 64, -1).T
    valid_idx = np.isfinite(flat).all(axis=1)
    proba = np.zeros(flat.shape[0])
    proba[valid_idx] = model.predict_proba(flat[valid_idx])[:, 1]
    prob_map = proba.reshape(shape)
    pred_map = (prob_map > 0.5).astype(np.uint8)

    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
        tmp_path = Path(tmp.name)
    with rasterio.open(tmp_path, 'w', driver='GTiff',
                       height=shape[0], width=shape[1],
                       count=1, dtype='uint8', crs=crs, transform=t) as dst:
        dst.write(pred_map, 1)

    try:
        geojson = raster_to_geojson(tmp_path, output_path=None, min_area_ha=0.5)
        month_map_est = estimate_month_map(stack)
        valid_feats = enrich_and_validate(geojson, month_map_est, t, crs)
        all_submission_features.extend(valid_feats)
        print(f'{tile_id}: {len(valid_feats):,} polygons')
    except ValueError:
        print(f'{tile_id}: 0 polygons (no deforestation detected)')
    finally:
        tmp_path.unlink(missing_ok=True)

final_geojson = {'type': 'FeatureCollection', 'features': all_submission_features}
out_path = SUBMISSION_DIR / 'submission.geojson'
with open(out_path, 'w') as f:
    json.dump(final_geojson, f)
print(f'\nSUBMISSION READY: {out_path}  ({len(all_submission_features)} total polygons)')
